# Mini-FORESIGHT — Step 6: Feature Engineering

In the previous notebooks we **understood**, **cleaned** and **explored** the data.
Now we **create features** that will let a machine-learning model learn from our historical sales.

Raw historical sales look like this:

```
date        sku_id   units_sold
2025-01-01  SKU003   4
2025-01-02  SKU003   5
2025-01-03  SKU003   3
2025-01-04  SKU003   6
```

A model cannot predict "tomorrow" from a single `units_sold` column. We need to give it **context** by creating features such as:

- **lag_1 / lag_2 / lag_3** — sales from 1, 2 and 3 days ago
- **rolling_mean_3 / rolling_mean_7** — recent average demand over 3 and 7 days
- **day_of_week** — which day of the week it is
- **is_weekend** — whether it is Saturday or Sunday

These features turn "today's prediction" into a question the model can answer using **yesterday's demand, 2-days-ago demand, 3-days-ago demand, recent average demand, and the day of week**.

We will **NOT** in this notebook:
- Train a machine learning model
- Evaluate forecasts
- Create a Streamlit app

We will only build the feature set that the forecasting step will use later.

import pandas as pd
from pathlib import Path

print("pandas version:", pd.__version__)

In [1]:
import pandas as pd
from pathlib import Path

print("pandas version:", pd.__version__)

pandas version: 3.0.5


In [2]:
# Load cleaned sales data

sales = pd.read_csv(
    Path("../data/processed/sales_daily_clean.csv")
)

print("Sales data loaded successfully")
print("Shape:", sales.shape)
print()
print(sales.head())

Sales data loaded successfully
Shape: (42, 3)

         date  sku_id  units_sold
0  2025-01-01  SKU001           2
1  2025-01-01  SKU002           3
2  2025-01-01  SKU003           4
3  2025-01-02  SKU001           1
4  2025-01-02  SKU002           4


In [3]:
# Convert date to datetime
sales["date"] = pd.to_datetime(sales["date"])

# Sort by SKU and date
sales = sales.sort_values(["sku_id", "date"]).reset_index(drop=True)

print("Date conversion and sorting completed.")
print()
print("Date dtype:", sales["date"].dtype)
print()
print(sales.head(10))

Date conversion and sorting completed.

Date dtype: datetime64[us]

        date  sku_id  units_sold
0 2025-01-01  SKU001           2
1 2025-01-02  SKU001           1
2 2025-01-03  SKU001           3
3 2025-01-04  SKU001           2
4 2025-01-05  SKU001           2
5 2025-01-06  SKU001           3
6 2025-01-07  SKU001           2
7 2025-01-08  SKU001           1
8 2025-01-09  SKU001           3
9 2025-01-10  SKU001           2


In [4]:
# Create lag-1 feature
# Previous day's sales for the same SKU

sales["lag_1"] = (
    sales.groupby("sku_id")["units_sold"]
    .shift(1)
)

print("lag_1 feature created.")
print()
print(sales[["date", "sku_id", "units_sold", "lag_1"]].head(10))

lag_1 feature created.

        date  sku_id  units_sold  lag_1
0 2025-01-01  SKU001           2    NaN
1 2025-01-02  SKU001           1    2.0
2 2025-01-03  SKU001           3    1.0
3 2025-01-04  SKU001           2    3.0
4 2025-01-05  SKU001           2    2.0
5 2025-01-06  SKU001           3    2.0
6 2025-01-07  SKU001           2    3.0
7 2025-01-08  SKU001           1    2.0
8 2025-01-09  SKU001           3    1.0
9 2025-01-10  SKU001           2    3.0


In [5]:
# Create lag-2 and lag-3 features
# Previous 2 and 3 days' sales for the same SKU

sales["lag_2"] = (
    sales.groupby("sku_id")["units_sold"]
    .shift(2)
)

sales["lag_3"] = (
    sales.groupby("sku_id")["units_sold"]
    .shift(3)
)

print("lag_2 and lag_3 features created.")
print()

print(
    sales[
        ["date", "sku_id", "units_sold", "lag_1", "lag_2", "lag_3"]
    ].head(10)
)

lag_2 and lag_3 features created.

        date  sku_id  units_sold  lag_1  lag_2  lag_3
0 2025-01-01  SKU001           2    NaN    NaN    NaN
1 2025-01-02  SKU001           1    2.0    NaN    NaN
2 2025-01-03  SKU001           3    1.0    2.0    NaN
3 2025-01-04  SKU001           2    3.0    1.0    2.0
4 2025-01-05  SKU001           2    2.0    3.0    1.0
5 2025-01-06  SKU001           3    2.0    2.0    3.0
6 2025-01-07  SKU001           2    3.0    2.0    2.0
7 2025-01-08  SKU001           1    2.0    3.0    2.0
8 2025-01-09  SKU001           3    1.0    2.0    3.0
9 2025-01-10  SKU001           2    3.0    1.0    2.0


In [6]:
# Create 3-day rolling average
# Uses only previous days' sales

sales["rolling_mean_3"] = (
    sales.groupby("sku_id")["units_sold"]
    .transform(
        lambda x: x.shift(1).rolling(window=3).mean()
    )
)

print("rolling_mean_3 feature created.")
print()

print(
    sales[
        [
            "date",
            "sku_id",
            "units_sold",
            "lag_1",
            "lag_2",
            "lag_3",
            "rolling_mean_3"
        ]
    ].head(10)
)

rolling_mean_3 feature created.

        date  sku_id  units_sold  lag_1  lag_2  lag_3  rolling_mean_3
0 2025-01-01  SKU001           2    NaN    NaN    NaN             NaN
1 2025-01-02  SKU001           1    2.0    NaN    NaN             NaN
2 2025-01-03  SKU001           3    1.0    2.0    NaN             NaN
3 2025-01-04  SKU001           2    3.0    1.0    2.0        2.000000
4 2025-01-05  SKU001           2    2.0    3.0    1.0        2.000000
5 2025-01-06  SKU001           3    2.0    2.0    3.0        2.333333
6 2025-01-07  SKU001           2    3.0    2.0    2.0        2.333333
7 2025-01-08  SKU001           1    2.0    3.0    2.0        2.333333
8 2025-01-09  SKU001           3    1.0    2.0    3.0        2.000000
9 2025-01-10  SKU001           2    3.0    1.0    2.0        2.000000


In [7]:
# Create 7-day rolling average
# Uses only previous days' sales

sales["rolling_mean_7"] = (
    sales.groupby("sku_id")["units_sold"]
    .transform(
        lambda x: x.shift(1).rolling(window=7).mean()
    )
)

print("rolling_mean_7 feature created.")
print()

print(
    sales[
        [
            "date",
            "sku_id",
            "units_sold",
            "lag_1",
            "lag_2",
            "lag_3",
            "rolling_mean_3",
            "rolling_mean_7"
        ]
    ].head(12)
)

rolling_mean_7 feature created.

         date  sku_id  units_sold  lag_1  lag_2  lag_3  rolling_mean_3  \
0  2025-01-01  SKU001           2    NaN    NaN    NaN             NaN   
1  2025-01-02  SKU001           1    2.0    NaN    NaN             NaN   
2  2025-01-03  SKU001           3    1.0    2.0    NaN             NaN   
3  2025-01-04  SKU001           2    3.0    1.0    2.0        2.000000   
4  2025-01-05  SKU001           2    2.0    3.0    1.0        2.000000   
5  2025-01-06  SKU001           3    2.0    2.0    3.0        2.333333   
6  2025-01-07  SKU001           2    3.0    2.0    2.0        2.333333   
7  2025-01-08  SKU001           1    2.0    3.0    2.0        2.333333   
8  2025-01-09  SKU001           3    1.0    2.0    3.0        2.000000   
9  2025-01-10  SKU001           2    3.0    1.0    2.0        2.000000   
10 2025-01-11  SKU001           2    2.0    3.0    1.0        2.000000   
11 2025-01-12  SKU001           3    2.0    2.0    3.0        2.333333   

    

In [8]:
# Create calendar features

sales["day_of_week"] = sales["date"].dt.dayofweek

sales["is_weekend"] = (
    sales["day_of_week"] >= 5
).astype(int)

print("Calendar features created.")
print()

print(
    sales[
        [
            "date",
            "sku_id",
            "units_sold",
            "day_of_week",
            "is_weekend"
        ]
    ].head(10)
)

Calendar features created.

        date  sku_id  units_sold  day_of_week  is_weekend
0 2025-01-01  SKU001           2            2           0
1 2025-01-02  SKU001           1            3           0
2 2025-01-03  SKU001           3            4           0
3 2025-01-04  SKU001           2            5           1
4 2025-01-05  SKU001           2            6           1
5 2025-01-06  SKU001           3            0           0
6 2025-01-07  SKU001           2            1           0
7 2025-01-08  SKU001           1            2           0
8 2025-01-09  SKU001           3            3           0
9 2025-01-10  SKU001           2            4           0


In [9]:
# Display the complete feature set

print("Feature engineering completed.")
print()
print("Shape:", sales.shape)
print()
print("Columns:")
print(sales.columns.tolist())
print()
print("First 10 rows:")
print(sales.head(10))

Feature engineering completed.

Shape: (42, 10)

Columns:
['date', 'sku_id', 'units_sold', 'lag_1', 'lag_2', 'lag_3', 'rolling_mean_3', 'rolling_mean_7', 'day_of_week', 'is_weekend']

First 10 rows:
        date  sku_id  units_sold  lag_1  lag_2  lag_3  rolling_mean_3  \
0 2025-01-01  SKU001           2    NaN    NaN    NaN             NaN   
1 2025-01-02  SKU001           1    2.0    NaN    NaN             NaN   
2 2025-01-03  SKU001           3    1.0    2.0    NaN             NaN   
3 2025-01-04  SKU001           2    3.0    1.0    2.0        2.000000   
4 2025-01-05  SKU001           2    2.0    3.0    1.0        2.000000   
5 2025-01-06  SKU001           3    2.0    2.0    3.0        2.333333   
6 2025-01-07  SKU001           2    3.0    2.0    2.0        2.333333   
7 2025-01-08  SKU001           1    2.0    3.0    2.0        2.333333   
8 2025-01-09  SKU001           3    1.0    2.0    3.0        2.000000   
9 2025-01-10  SKU001           2    3.0    1.0    2.0        2.000000  

In [10]:
# Check missing values created by lag and rolling features

print("Missing values in engineered features:")
print()

print(sales.isna().sum())

Missing values in engineered features:

date               0
sku_id             0
units_sold         0
lag_1              3
lag_2              6
lag_3              9
rolling_mean_3     9
rolling_mean_7    21
day_of_week        0
is_weekend         0
dtype: int64


In [11]:
# Remove rows that do not have enough historical data
# We need 7 previous days for rolling_mean_7

model_data = sales.dropna().reset_index(drop=True)

print("Rows before removing NaN values:", len(sales))
print("Rows after removing NaN values:", len(model_data))
print()

print("Remaining data by SKU:")
print(model_data["sku_id"].value_counts().sort_index())

Rows before removing NaN values: 42
Rows after removing NaN values: 21

Remaining data by SKU:
sku_id
SKU001    7
SKU002    7
SKU003    7
Name: count, dtype: int64


In [12]:
# Validate the final feature dataset

print("=== Final Feature Dataset Validation ===")
print()

print("Shape:", model_data.shape)
print()

print("Missing values:")
print(model_data.isna().sum())
print()

print("Rows per SKU:")
print(model_data["sku_id"].value_counts().sort_index())
print()

print("Date range:")
print(model_data["date"].min().date(), "to", model_data["date"].max().date())

=== Final Feature Dataset Validation ===

Shape: (21, 10)

Missing values:
date              0
sku_id            0
units_sold        0
lag_1             0
lag_2             0
lag_3             0
rolling_mean_3    0
rolling_mean_7    0
day_of_week       0
is_weekend        0
dtype: int64

Rows per SKU:
sku_id
SKU001    7
SKU002    7
SKU003    7
Name: count, dtype: int64

Date range:
2025-01-08 to 2025-01-14


In [13]:
# Save the engineered feature dataset

output_path = Path("../data/processed/sales_features.csv")

model_data.to_csv(output_path, index=False)

print("Feature dataset saved successfully.")
print()
print("Saved to:", output_path)
print("Rows:", len(model_data))
print("Columns:", len(model_data.columns))

Feature dataset saved successfully.

Saved to: ..\data\processed\sales_features.csv
Rows: 21
Columns: 10


In [14]:
# Reload the saved feature dataset and verify it

features_check = pd.read_csv(
    Path("../data/processed/sales_features.csv")
)

print("=== Saved Feature Dataset Verification ===")
print()

print("Shape:", features_check.shape)
print()

print("Columns:")
print(features_check.columns.tolist())
print()

print("Missing values:")
print(features_check.isna().sum())
print()

print("First 5 rows:")
print(features_check.head())

=== Saved Feature Dataset Verification ===

Shape: (21, 10)

Columns:
['date', 'sku_id', 'units_sold', 'lag_1', 'lag_2', 'lag_3', 'rolling_mean_3', 'rolling_mean_7', 'day_of_week', 'is_weekend']

Missing values:
date              0
sku_id            0
units_sold        0
lag_1             0
lag_2             0
lag_3             0
rolling_mean_3    0
rolling_mean_7    0
day_of_week       0
is_weekend        0
dtype: int64

First 5 rows:
         date  sku_id  units_sold  lag_1  lag_2  lag_3  rolling_mean_3  \
0  2025-01-08  SKU001           1    2.0    3.0    2.0        2.333333   
1  2025-01-09  SKU001           3    1.0    2.0    3.0        2.000000   
2  2025-01-10  SKU001           2    3.0    1.0    2.0        2.000000   
3  2025-01-11  SKU001           2    2.0    3.0    1.0        2.000000   
4  2025-01-12  SKU001           3    2.0    2.0    3.0        2.333333   

   rolling_mean_7  day_of_week  is_weekend  
0        2.142857            2           0  
1        2.000000        